<a href="https://colab.research.google.com/github/MachineLearnia/Python-Machine-Learning/blob/master/18%20-%20Pandas%20et%20S%C3%A9ries%20Temporelles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 18/30 Pandas et séries temporelles

## 1. Travailler avec des séries Temporelles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
bitcoin = pd.read_csv("Dataset/BTC-EUR.csv")
bitcoin.head()


In [ ]:
bitcoin = pd.read_csv('Dataset/BTC-EUR.csv', index_col='Date', parse_dates=True)
print(bitcoin.head(3))


In [ ]:
# pd.set_option("display.max_columns", None)  # affiche toutes les colonnes
pd.set_option(
    "display.width", 2000
)  # évite les retours à la ligne
print(bitcoin.tail(3))


In [ ]:
btc = bitcoin.reset_index()
btc[:-1]


In [ ]:
bitcoin["Close"].plot(figsize=(9, 6))
plt.show()


In [ ]:
bitcoin["Volume"][1800:-1].plot()
plt.show()


In [ ]:
bitcoin.index


In [ ]:
bitcoin.loc['2019','Close'].plot(c='r', linewidth=5, alpha=.5)
# OU
bitcoin.loc['2019']['Close'].plot()
plt.tight_layout()
plt.grid()


In [ ]:
bitcoin.loc['2017':'2019']["Close"].plot()
plt.tight_layout()
plt.grid()


In [ ]:
bitcoin["2017":"2019"]["Close"].plot()
plt.tight_layout()
plt.grid()


In [ ]:
bitcoin.describe()


## 2. Resample

In [ ]:
bitcoin.loc['2019', 'Close'].resample('ME').mean().plot()
plt.show()


In [ ]:
monthly = bitcoin.loc["2019", "Close"].resample("ME").mean()

for i, (date, value) in enumerate(monthly.items()):
    plt.plot(date, value, marker="o", color=plt.cm.tab10(i % 10))  # palette tab10


In [ ]:
bitcoin["2019":]["Close"].plot()
plt.tight_layout()
plt.grid()


In [ ]:
daily = bitcoin.loc["2019":, "Close"]

for i, (month, segment) in enumerate(daily.groupby(daily.index.month)):
    plt.plot(segment.index, segment.values, color=plt.cm.tab10(i % 10), linewidth=2)

plt.xticks(
    ticks=monthly.index.to_period("M").to_timestamp(),
    labels=monthly.index.strftime("%b"),  # noms abrégés des mois
    rotation=45,
)
plt.grid()
plt.tight_layout()


In [ ]:
monthly = bitcoin.loc["2019", "Close"].resample("ME").mean()

# Boucler sur les paires de points consécutifs
for i in range(len(monthly) - 1):
    x = [monthly.index[i], monthly.index[i + 1]]
    y = [monthly.values[i], monthly.values[i + 1]]
    plt.plot(x, y, color=plt.cm.tab10(i % 10), linewidth=2)

plt.grid()
plt.tight_layout()


In [ ]:
colors = plt.cm.viridis(np.linspace(0, 1, len(monthly) - 1))
for i in range(len(monthly) - 1):
    plt.plot(
        monthly.index[i : i + 2],
        monthly.values[i : i + 2],
        color=colors[i],
        linewidth=2,
    )


In [ ]:
# Série journalière
daily = bitcoin.loc["2019", "Close"]

# Série mensuelle (pour récupérer les bornes et couleurs)
monthly = daily.resample("MS").mean()

# Boucler sur chaque mois
for i in range(len(monthly)-1):
    start = monthly.index[i]
    end = monthly.index[i+1]
    segment = daily.loc[start:end]  # sous-série journalière
    plt.plot(segment.index, segment.values, color=plt.cm.tab10(i % 10), linewidth=2)
    
plt.xticks(
    ticks=monthly.index.to_period("M").to_timestamp(),
    labels=monthly.index.strftime("%b"),
    rotation=45,
)
plt.grid()
plt.tight_layout()


In [ ]:
monthly = bitcoin.loc["2019", "Close"].resample("ME").mean()

monthly.plot(kind="bar", colormap="tab20", figsize=(10, 5))
plt.tight_layout()
plt.grid()


In [ ]:
monthly = bitcoin.loc["2019", "Close"].resample("ME").mean()

plt.scatter(monthly.index, monthly.values, c=range(len(monthly)), cmap="viridis")
plt.plot(monthly.index, monthly.values, alpha=0.3, color="gray")  # relier les points
plt.colorbar(label="Mois")


In [ ]:
bitcoin.loc['2019', 'Close'].resample('2W').mean().plot()
plt.show()


In [ ]:
bitcoin.loc['2019', 'Close'].resample('2W').std().plot()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
bitcoin.loc['2019', 'Close'].plot()
bitcoin.loc['2019', 'Close'].resample('ME').mean().plot(label='moyenne par mois', lw=3, ls=':', alpha=0.8)
bitcoin.loc['2019', 'Close'].resample('W').mean().plot(label='moyenne par semaine', lw=2, ls='--', alpha=0.8)
plt.legend()
plt.show()


## 3. Aggregate

In [ ]:
bitcoin.loc['2019',"Close"].resample("W").agg(["mean", "std", "min", "max"])


In [ ]:
m = bitcoin['Close'].resample('W').agg(['mean', 'std', 'min', 'max'])
plt.figure(figsize=(12, 8))
m['mean']['2019'].plot(label='moyenne par semaine')
plt.fill_between(m.index, m['max'], m['min'], alpha=0.2, label='min-max par semaine')

plt.legend()
plt.show()


In [ ]:
bitcoin.loc['2019', 'Close'].resample('W').agg(['mean', 'std', 'min', 'max']).plot()


## 4. Moving Average (Rolling) et EWM

## Moving Average (Moyenne mobile)

In [ ]:
bitcoin.loc['2019', 'Close'].plot()
bitcoin.loc['2019', 'Close'].rolling(window=7).mean().plot(label='Moving average', lw=3, ls=':')
plt.legend()


## Exponential Weighted Average (Moyenne mobile exponentielle)

In [ ]:
plt.figure(figsize=(12, 8))
bitcoin.loc['2019-09', 'Close'].plot()
bitcoin.loc['2019-09', 'Close'].rolling(window=7).mean().plot(label='non centre', lw=3, ls=':', alpha=0.8)
bitcoin.loc['2019-09', 'Close'].rolling(window=7, center=True).mean().plot(label='centre', lw=3, ls=':', alpha=0.8)
bitcoin.loc['2019-09', 'Close'].ewm(alpha=0.6).mean().plot(label='ewm', lw=3, ls=':', alpha=0.8)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
bitcoin.loc['2019-09', 'Close'].plot()
for i in np.arange(0.2, 1, 0.2):
    bitcoin.loc['2019-09', 'Close'].ewm(alpha=i).mean().plot(label=f'ewm {i:.1f}', ls='--', alpha=0.8)
plt.legend()
plt.show()


## 5. Comparaison de 2 série temporelles = Assembler des Datasets

In [ ]:
ethereum = pd.read_csv('Dataset/ETH-EUR.csv', index_col='Date', parse_dates=True)
print(ethereum.shape)
ethereum.head(3)


In [ ]:
ethereum.loc['2019']['Close'].plot()


In [ ]:
btc_eth = pd.merge(bitcoin, ethereum, on='Date', how='inner', suffixes=('_btc', '_eth'))
print(bitcoin.shape)
print(ethereum.shape)
print(btc_eth.shape)
btc_eth.head(3)


In [ ]:
# btc_eth.loc["2015":, "Close_btc"].plot()
# btc_eth.loc["2015":]["Close_eth"].plot()
btc_eth.loc[:, ["Close_btc" , "Close_eth"]].plot()


In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# Axe de gauche pour BTC
ax1.plot(btc_eth.index, btc_eth["Close_btc"], color="tab:orange", label="BTC")
ax1.set_ylabel("BTC Close", color="tab:orange")
ax1.tick_params(axis="y", labelcolor="tab:orange")

# Axe de droite pour ETH
ax2 = ax1.twinx()
ax2.plot(btc_eth.index, btc_eth["Close_eth"], color="tab:blue", label="ETH")
ax2.set_ylabel("ETH Close", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")

# Titre et grille
plt.title("BTC vs ETH (cours de clôture)")
fig.tight_layout()
plt.grid()
plt.show()


In [ ]:
btc_eth.loc[:, ["Close_btc", "Close_eth"]].plot(subplots=True, figsize=(12, 8))


In [ ]:
btc_eth.loc[:, ["Close_btc", "Close_eth"]].describe()


In [ ]:
btc_eth.loc[:, ["Close_btc", "Close_eth"]].corr()


## 6. Exercice et Solution

In [ ]:
btc = bitcoin.copy()


In [ ]:
btc.loc["2019":, "Close"].diff().plot()
plt.title('Volatilité du BTC')
plt.show()


In [ ]:
btc.loc["2019-06":'2019-07', "Close"].diff().plot()
plt.title("Volatilité du BTC")
plt.show()


In [ ]:
btc.tail(3)


In [ ]:
btc.loc["2019", "Close"].head(3)


## Stratégie de la Tortue

In [ ]:
data = bitcoin.copy()
data['Buy'] = np.zeros(len(data))
data['Sell'] = np.zeros(len(data))


In [ ]:
data['RollingMax'] = data['Close'].shift(1).rolling(window=28).max()
data['RollingMin'] = data['Close'].shift(1).rolling(window=28).min()
data.loc[data['RollingMax'] < data['Close'], 'Buy'] = 1
data.loc[data['RollingMin'] > data['Close'], 'Sell'] = -1


In [ ]:
start ='2019'
end='2019'
fig, ax = plt.subplots(2, figsize=(12, 8), sharex=True)
#plt.figure(figsize=(12, 8))
#plt.subplot(211)
ax[0].plot(data['Close'][start:end])
ax[0].plot(data['RollingMin'][start:end])
ax[0].plot(data['RollingMax'][start:end])
ax[0].legend(['close', 'min', 'max'])
ax[1].plot(data['Buy'][start:end], c='green')
ax[1].plot(data['Sell'][start:end], c='tab:red')
ax[1].legend(['buy', 'sell'])

# plt.tight_layout()
plt.show()


In [ ]:
%pip install yfinance


In [ ]:
import yfinance as yf

df = yf.download("BTC-USD")
df.to_csv("BTC.csv")

df.sample(5)


In [ ]:
df.tail(1)


In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

# Télécharger plusieurs tickers
tickers = ["BTC-USD", "ETH-USD"]
# df = yf.download(tickers, start="2019-01-01", end="2025-11-25")
df = yf.download(tickers, period="max", auto_adjust=True) # True default

# Aperçu des données
print(df.head())

# Tracer les cours de clôture
fig, ax1 = plt.subplots(figsize=(12, 6))


# Axe de gauche pour BTC
ax1.plot(df.index, df["Close"]["BTC-USD"], color="tab:orange", label="BTC")
ax1.set_ylabel("BTC Close", color="tab:orange")
ax1.tick_params(axis="y", labelcolor="tab:orange")

# Axe de droite pour ETH
ax2 = ax1.twinx()
ax2.plot(df.index, df["Close"]["ETH-USD"], color="tab:blue", label="ETH")
ax2.set_ylabel("ETH Close", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")

# Titre et grille
plt.title("BTC vs ETH (cours de clôture)")

plt.grid()
plt.show()


In [ ]:
print(df.columns)


In [ ]:
df.shape


In [ ]:
df['Close']['ETH-USD'].tail


In [ ]:
df['Close'].tail(5)


In [ ]:
df["Close"].describe()


In [ ]:
df["Close"].iloc[-1]


In [ ]:
10/3, round(10 / 3, 15)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(6, 15))

# Load the example car crash dataset
crashes = sns.load_dataset("car_crashes").sort_values("total", ascending=False)

# Plot the total crashes
sns.set_color_codes("pastel")
sns.barplot(x="total", y="abbrev", data=crashes, label="Total", color="b")

# Plot the crashes where alcohol was involved
sns.set_color_codes("muted")
sns.barplot(x="alcohol", y="abbrev", data=crashes, label="Alcohol-involved", color="b")

# Add a legend and informative axis label
ax.legend(ncol=2, loc="lower right", frameon=True)
ax.set(xlim=(0, 24), ylabel="", xlabel="Automobile collisions per billion miles")
sns.despine(left=True, bottom=True)
